In [1]:
import pandas as pd
from functions.eval import *

In [2]:
model_pred_col = "MarBERT"
model_name = "IbrahimAmin/marbertv2-finetuned-egyptian-hate-speech-detection"

In [3]:
lime_eval = pd.read_csv("data/xai_eval_hate/xai_eval_hate_lime_" + model_pred_col + ".csv")
shap_eval = pd.read_csv("data/xai_eval_hate/xai_eval_hate_shap_" + model_pred_col + ".csv")
ig_eval = pd.read_csv("data/xai_eval_hate/xai_eval_hate_ig_" + model_pred_col + ".csv")
dl_eval = pd.read_csv("data/xai_eval_hate/xai_eval_hate_dl_" + model_pred_col + ".csv")
ensemble_eval = pd.read_csv("data/xai_eval_hate/xai_eval_hate_ensemble_" + model_pred_col + ".csv")

In [4]:
eval_df = pd.concat([lime_eval, shap_eval, ig_eval, dl_eval, ensemble_eval], axis=1)

In [5]:
XAI_methods = ["LIME", "SHAP", "IG", "DeepLIFT", "EXAI_LIME_SHAP_IG_DL_mean", 
               "EXAI_LIME_SHAP_IG_DL_median", "EXAI_LIME_SHAP_IG_mean", 
               "EXAI_LIME_SHAP_IG_median", "EXAI_LIME_SHAP_mean"]
metrics = ["comprehensiveness", "sufficiency", "corr_loo", "ins_AUC", "del_AUC", "combined"]

In [6]:
df = pd.DataFrame(
    {metric: [] for metric in metrics},
)

In [7]:
for xai_method in XAI_methods:
    for metric in metrics:
        col_name = f"{xai_method}_{metric}"
        if col_name in eval_df.columns:
            df.loc[xai_method, metric] = eval_df[col_name].mean()
        else:
            df.loc[xai_method, metric] = None

In [8]:
borda_count(df)

,comprehensiveness,sufficiency,corr_loo,ins_AUC,del_AUC,combined,borda
LIME,0.572074,0.068594,0.239918,0.836551,0.576242,0.676028,29
SHAP,0.408361,0.067538,0.259267,0.857844,0.537648,0.656608,26
IG,0.308981,0.105573,0.156478,0.829471,0.599485,0.599637,41
DeepLIFT,0.432265,0.152298,-0.024368,0.719654,0.753213,0.543196,43
EXAI_LIME_SHAP_IG_DL_mean,0.454866,0.057373,0.257062,0.855077,0.548746,0.667976,24
EXAI_LIME_SHAP_IG_DL_median,0.473447,0.059695,0.258411,0.853582,0.548910,0.671155,25
EXAI_LIME_SHAP_IG_mean,0.483436,0.047411,0.288703,0.862900,0.521636,0.683402,7
EXAI_LIME_SHAP_IG_median,0.438400,0.061986,0.287806,0.860142,0.526900,0.669438,17
EXAI_LIME_SHAP_mean,0.507874,0.055887,0.284753,0.858473,0.531426,0.684589,13


In [9]:
df_std = df.copy()
for xai_method in XAI_methods:
    for metric in metrics:
        col_name = f"{xai_method}_{metric}"
        if col_name in eval_df.columns:
            df_std.loc[xai_method, metric] = eval_df[col_name].std()
        else:
            df_std.loc[xai_method, metric] = None

In [10]:
df_std.drop(columns=["borda"], inplace=True)
df_std

,comprehensiveness,sufficiency,corr_loo,ins_AUC,del_AUC,combined
LIME,0.437956,0.241822,0.338164,0.123780,0.240947,0.136411
SHAP,0.442574,0.243428,0.320849,0.111266,0.233492,0.144737
IG,0.432163,0.281416,0.344182,0.146113,0.229019,0.147160
DeepLIFT,0.462782,0.324200,0.339231,0.194408,0.183474,0.155986
EXAI_LIME_SHAP_IG_DL_mean,0.454200,0.223640,0.303149,0.118934,0.235341,0.136188
EXAI_LIME_SHAP_IG_DL_median,0.454196,0.223811,0.305745,0.118668,0.235815,0.134717
EXAI_LIME_SHAP_IG_mean,0.450468,0.207571,0.315867,0.113134,0.235972,0.135495
EXAI_LIME_SHAP_IG_median,0.448603,0.231830,0.318987,0.112507,0.235433,0.140079
EXAI_LIME_SHAP_mean,0.448765,0.227445,0.315080,0.113306,0.238937,0.138096


Spearman rank correlation between the Combined metric and Borda count rankings

In [11]:
from scipy import stats

In [12]:
corr, pval = stats.spearmanr(df["combined"], df["borda"])
corr, pval

(np.float64(-0.7666666666666667), np.float64(0.01594401657897401))

Leave-one-metric-out

In [13]:
metric_to_leave_out = "comprehensiveness"
df["combined"] = df.apply(lambda row: combined_metric(comp=None, suff=row["sufficiency"], corr_loo=row["corr_loo"], ins_auc=row["ins_AUC"], del_auc=row["del_AUC"]), axis=1)
borda_count_leave_one_out(df.drop(columns=[metric_to_leave_out]), metric_to_leave_out=metric_to_leave_out)

,sufficiency,corr_loo,ins_AUC,del_AUC,combined,borda
LIME,0.068594,0.239918,0.836551,0.576242,0.702919,28
SHAP,0.067538,0.259267,0.857844,0.537648,0.720573,18
IG,0.105573,0.156478,0.829471,0.599485,0.675663,32
DeepLIFT,0.152298,-0.024368,0.719654,0.753213,0.575490,36
EXAI_LIME_SHAP_IG_DL_mean,0.057373,0.257062,0.855077,0.548746,0.719372,19
EXAI_LIME_SHAP_IG_DL_median,0.059695,0.258411,0.853582,0.548910,0.718546,21
EXAI_LIME_SHAP_IG_mean,0.047411,0.288703,0.862900,0.521636,0.734551,4
EXAI_LIME_SHAP_IG_median,0.061986,0.287806,0.860142,0.526900,0.728789,11
EXAI_LIME_SHAP_mean,0.055887,0.284753,0.858473,0.531426,0.728384,11


In [14]:
metric_to_leave_out = "sufficiency"
df["combined"] = df.apply(lambda row: combined_metric(comp=row["comprehensiveness"], suff=None, corr_loo=row["corr_loo"], ins_auc=row["ins_AUC"], del_auc=row["del_AUC"]), axis=1)
borda_count_leave_one_out(df.drop(columns=[metric_to_leave_out]), metric_to_leave_out=metric_to_leave_out)

,comprehensiveness,corr_loo,ins_AUC,del_AUC,combined,borda
LIME,0.572074,0.239918,0.836551,0.576242,0.613085,22
SHAP,0.408361,0.259267,0.857844,0.537648,0.589548,20
IG,0.308981,0.156478,0.829471,0.599485,0.529302,33
DeepLIFT,0.432265,-0.024368,0.719654,0.753213,0.471630,34
EXAI_LIME_SHAP_IG_DL_mean,0.454866,0.257062,0.855077,0.548746,0.597432,21
EXAI_LIME_SHAP_IG_DL_median,0.473447,0.258411,0.853582,0.548910,0.601831,21
EXAI_LIME_SHAP_IG_mean,0.483436,0.288703,0.862900,0.521636,0.617263,6
EXAI_LIME_SHAP_IG_median,0.438400,0.287806,0.860142,0.526900,0.603886,12
EXAI_LIME_SHAP_mean,0.507874,0.284753,0.858473,0.531426,0.619324,11


In [15]:
metric_to_leave_out = "corr_loo"
df["combined"] = df.apply(lambda row: combined_metric(comp=row["comprehensiveness"], suff=row["sufficiency"], corr_loo=None, ins_auc=row["ins_AUC"], del_auc=row["del_AUC"]), axis=1)
borda_count_leave_one_out(df.drop(columns=[metric_to_leave_out]), metric_to_leave_out=metric_to_leave_out)

,comprehensiveness,sufficiency,ins_AUC,del_AUC,combined,borda
LIME,0.572074,0.068594,0.836551,0.576242,0.690947,22
SHAP,0.408361,0.067538,0.857844,0.537648,0.665255,22
IG,0.308981,0.105573,0.829471,0.599485,0.608349,33
DeepLIFT,0.432265,0.152298,0.719654,0.753213,0.561602,34
EXAI_LIME_SHAP_IG_DL_mean,0.454866,0.057373,0.855077,0.548746,0.675956,18
EXAI_LIME_SHAP_IG_DL_median,0.473447,0.059695,0.853582,0.548910,0.679606,20
EXAI_LIME_SHAP_IG_mean,0.483436,0.047411,0.862900,0.521636,0.694322,6
EXAI_LIME_SHAP_IG_median,0.438400,0.061986,0.860142,0.526900,0.677414,15
EXAI_LIME_SHAP_mean,0.507874,0.055887,0.858473,0.531426,0.694759,10


In [16]:
metric_to_leave_out = "ins_AUC"
df["combined"] = df.apply(lambda row: combined_metric(comp=row["comprehensiveness"], suff=row["sufficiency"], corr_loo=row["corr_loo"], ins_auc=None, del_auc=row["del_AUC"]), axis=1)
borda_count_leave_one_out(df.drop(columns=[metric_to_leave_out]), metric_to_leave_out=metric_to_leave_out)

,comprehensiveness,sufficiency,corr_loo,del_AUC,combined,borda
LIME,0.572074,0.068594,0.239918,0.576242,0.636799,22
SHAP,0.408361,0.067538,0.259267,0.537648,0.608202,22
IG,0.308981,0.105573,0.156478,0.599485,0.545541,33
DeepLIFT,0.432265,0.152298,-0.024368,0.753213,0.503642,34
EXAI_LIME_SHAP_IG_DL_mean,0.454866,0.057373,0.257062,0.548746,0.619320,19
EXAI_LIME_SHAP_IG_DL_median,0.473447,0.059695,0.258411,0.548910,0.623512,19
EXAI_LIME_SHAP_IG_mean,0.483436,0.047411,0.288703,0.521636,0.639685,6
EXAI_LIME_SHAP_IG_median,0.438400,0.061986,0.287806,0.526900,0.623354,15
EXAI_LIME_SHAP_mean,0.507874,0.055887,0.284753,0.531426,0.640734,10


In [17]:
metric_to_leave_out = "del_AUC"
df["combined"] = df.apply(lambda row: combined_metric(comp=row["comprehensiveness"], suff=row["sufficiency"], corr_loo=row["corr_loo"], ins_auc=row["ins_AUC"], del_auc=None), axis=1)
borda_count_leave_one_out(df.drop(columns=[metric_to_leave_out]), metric_to_leave_out=metric_to_leave_out)

,comprehensiveness,sufficiency,corr_loo,ins_AUC,combined,borda
LIME,0.572074,0.068594,0.239918,0.836551,0.739997,22
SHAP,0.408361,0.067538,0.259267,0.857844,0.707075,22
IG,0.308981,0.105573,0.156478,0.829471,0.652780,33
DeepLIFT,0.432265,0.152298,-0.024368,0.719654,0.621859,34
EXAI_LIME_SHAP_IG_DL_mean,0.454866,0.057373,0.257062,0.855077,0.720275,19
EXAI_LIME_SHAP_IG_DL_median,0.473447,0.059695,0.258411,0.853582,0.724135,19
EXAI_LIME_SHAP_IG_mean,0.483436,0.047411,0.288703,0.862900,0.735819,6
EXAI_LIME_SHAP_IG_median,0.438400,0.061986,0.287806,0.860142,0.720114,15
EXAI_LIME_SHAP_mean,0.507874,0.055887,0.284753,0.858473,0.738209,10
